# Lab: Kokoro ← Stage 1–2 synthesis manifest

**Goal:** Check TTS compatibility with prepared pipeline output (`spoken_text`, `voice_profile_id`, multi-speaker segments).

**Not:** final model selection · production adapter · silent script edits

Runtime: Colab GPU (L4/A100 preferred).

## 0. Config — set your repo URL

In [ ]:
# @title Repo settings
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"


## 1. Clone repo

In [ ]:
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull
%cd {WORKDIR}
print("CWD:", os.getcwd())

## 2. Install control-plane + Kokoro

Stage 1–2 package is local editable install. Kokoro needs `espeak-ng`.

In [ ]:
!pip -q install -e ".[dev]" kokoro soundfile
!apt-get -qq update && apt-get -qq install -y espeak-ng

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 3. Stage 1–2: build synthesis manifest

This is the **same contract** production will use later.

In [ ]:
!mkdir -p outputs lab_audio/kokoro_part1
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json

import json
from pathlib import Path

m = json.loads(Path("outputs/part1_manifest.json").read_text())
print("transcript_id:", m["transcript_id"])
print("valid:", m["validation"]["valid"])
print("n_requests:", len(m["requests"]))
print("voice profiles:", [(e["speaker_id"], e["voice_profile_id"]) for e in m["speaker_map"]["entries"]])
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006"}:
        print(u["event_id"], "display:", u["display_text"])
        print("       spoken :", u["spoken_text"])

## 4. Lab render (Kokoro consumes manifest)

Maps `voice_profile_id` → Kokoro fixed IDs via `configs/voice_maps/kokoro_en_gb_part1.json`.

Uses **`spoken_text`** (Stage 2), not raw script rewrite.

In [ ]:
!python scripts/lab_render_kokoro_from_manifest.py \
  --manifest outputs/part1_manifest.json \
  --voice-map configs/voice_maps/kokoro_en_gb_part1.json \
  --out-dir lab_audio/kokoro_part1

report = json.loads(Path("lab_audio/kokoro_part1/lab_render_report.json").read_text())
print("ok:", report["ok_count"], "fail:", report["fail_count"])
print("notes:", report["compatibility_notes"])
for s in report["segments"]:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error"))

## 5. Listen in notebook

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

wavs = sorted(Path("lab_audio/kokoro_part1").glob("*.wav"))
print(f"{len(wavs)} wav files")
for w in wavs:
    print("===", w.name)
    display(Audio(filename=str(w)))

## 6. Optional ablation: feed display_text instead of spoken_text

If this sounds **worse** on spelling/postcode, Stage-2 normalisation is doing useful work.

In [ ]:
!python scripts/lab_render_kokoro_from_manifest.py \
  --manifest outputs/part1_manifest.json \
  --voice-map configs/voice_maps/kokoro_en_gb_part1.json \
  --out-dir lab_audio/kokoro_part1_display_text \
  --use-display-text \
  --limit 6

## 7. Human checklist

Open `docs/research/lab/checklist-stage1-2-compatibility.md` and mark:

1. Two speakers distinguishable?
2. Spelling (e004) clear with spoken_text?
3. Postcode (e006) usable?
4. Any crash / empty audio?
5. Did we need to edit the transcript script? (**must be no** — report issues instead)

Verdict: **Pass lab / Borderline / Fail lab** — still **not** final TTS selection.